# Hi-EF Phase 2: canonical residual matrix

This notebook runs the frozen `context`, `affect`, `interaction`, and `both` variants for all five model seeds in one auditable job. It uses train and validation only; the test partition is never loaded. Attach `ptrnghieu/hi-ef-features-v2`, enable a T4 GPU and Internet, then choose **Save Version → Save & Run All**.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/canonical_residual_matrix')

if not (REPO / '.git').exists():
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)
else:
    subprocess.run([
        'git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'
    ], check=True)

assert (FEATURES / '01_00059.pt').is_file(), 'Feature dataset is not attached'
MANIFEST = REPO / 'experiments/manifests/source_folder_split_seed42.csv'
assert MANIFEST.is_file()
commit = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
print('Ready at commit:', commit)

In [ ]:
command = [
    'python', str(REPO / 'experiments/run_canonical_residual_matrix.py'),
    '--manifest', str(MANIFEST),
    '--features-dir', str(FEATURES),
    '--output-dir', str(OUTPUT),
]
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd

summary_path = OUTPUT / 'canonical_residual_matrix_summary.json'
table_path = OUTPUT / 'canonical_residual_matrix.csv'
summary = json.loads(summary_path.read_text())
assert summary['test_evaluated'] is False
assert summary['method_selection_permitted'] is False
assert summary['partitions_touched'] == ['train', 'validation']
assert len(summary['runs']) == 5
assert all(len(runs) == 4 for runs in summary['runs'].values())
display(pd.read_csv(table_path))
print(json.dumps(summary['aggregates'], indent=2))
print('Download:', summary_path)
print('Download:', table_path)